<a href="https://colab.research.google.com/github/falamiieu2024-lgtm/ML-fundamentals-2026/blob/main/assignment_1_%3CFaris_Alami%5B_Alami%5D%3E.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Assignment 1 — Machine Learning Foundations: Data Preparation

**Dataset:** UCI Bank Marketing Dataset  

---

## Install & Import

In [1]:
import sys, importlib
def ensure(pkg, import_name=None):
    try:
        importlib.import_module(import_name or pkg)
    except ImportError:
        import subprocess
        subprocess.run([sys.executable, '-m', 'pip', 'install', pkg, '-q'], check=True)
        importlib.invalidate_caches()
for pkg, name in [('seaborn',None),('scikit-learn','sklearn'),('imbalanced-learn','imblearn'),('matplotlib',None),('pandas',None),('numpy',None)]:
    ensure(pkg, name)
print('All packages ready.')


All packages ready.


In [2]:
import warnings, io
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OrdinalEncoder, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, ConfusionMatrixDisplay
from sklearn.feature_selection import VarianceThreshold
from imblearn.over_sampling import SMOTE
warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')
pd.set_option('display.max_columns', 50)
RANDOM_STATE = 42
print('Imports complete.')


Imports complete.


## Load Dataset

In [ ]:
# Upload bank-additional.csv when prompted
from google.colab import files
uploaded = files.upload()

import io, pandas as pd
filename = list(uploaded.keys())[0]
df = pd.read_csv(io.BytesIO(uploaded[filename]), sep=';')
print(f'Loaded: {df.shape[0]} rows, {df.shape[1]} columns')
df.head()


---
## Task 0 — Task Ordering



The assignment lists tasks alphabetically, but the correct execution order is determined by one strict principle from our lectures on pipeline discipline: **any transformation that learns parameters from data must be fit on the training set only.**

### Chosen Pipeline Order

| # | Task | Reason |
|---|------|---------|
| 1 | **Identify the Prediction Target** | Pure inspection — defines the problem before any data is touched. |
| 2 | **Data Loading and Exploration** | EDA on the full dataset is allowed — we only observe, we learn no parameters. |
| 3 | **Managing Missing Values (strategy)** | We decide *how* to handle missing values but do not fit imputers yet. |
| 4 | **Data Splitting** | Must happen **before** any parameterised transformation. |
| 5 | **Managing Missing Values (fit & transform)** | Imputers fitted on `X_train` only, applied to all splits. |
| 6 | **Encoding Categorical Variables** | Encoders fitted on `X_train` only. |
| 7 | **Feature Scaling** | Scalers fitted on `X_train` only. |
| 8 | **Feature Selection** | Variance and correlations computed on `X_train` only. |
| 9 | **Addressing Class Imbalance** | SMOTE applied to `X_train` only — never to validation or test. |
| 10 | **Training a Logistic Regression Model** | Trained on resampled train, evaluated on validation. |

### Why This Order Prevents Leakage

- **Split before fit**: If we scaled before splitting, the scaler would absorb test-set statistics. The model would effectively have peeked at the test distribution.
- **Resample after split**: Applying SMOTE before splitting would allow synthetic minority samples (derived from real test examples) to appear in training, artificially inflating recall on the test set.
- **Feature selection on train only**: Selecting features based on variance or correlation computed on the whole dataset leaks test-set structure into the pipeline.

### Incorrect Ordering Example

A common mistake highlighted in Lecture 8 is to scale the entire dataset before splitting. The scaler's mean and standard deviation incorporate validation and test values. Standardization then maps test points relative to a distribution they helped define. This is classic **Preprocessing Leakage**: the test set is no longer truly unseen, producing optimistic metrics that will fail in production.

---
## Task 1 — Identifying the Prediction Target

> *Lecture material: Lecture 1 (Problem Formulation), Lecture 2 (Data Inspection).*

### Target Variable

The target is **y**  a binary indicator of whether a client subscribed to a term deposit (yes/no). The business problem is a **binary classification** task: given a client's profile and the campaign context before the call is made, predict the probability of subscription. This framing matters because it constrains which features are legitimate inputs, anything only observable after the call cannot be used.
### Why **y** and Not Something Else

Every other column describes either the client's characteristics or the campaign's process. They are inputs to the model, not outcomes of it. **y** is the only column that records what the bank is actually trying to predict: the client's decision.
### Variables That Could Superficially Appear as Targets

**duration** (call duration in seconds): Strongly correlated with subscription (~0.40) but only known after the call ends. At prediction time — before contact — this value does not exist. Using it would be a classic case of **Target Leakage**: the model would learn from information it cannot have in deployment, producing optimistic metrics that collapse entirely in production. The dataset authors explicitly flag it for removal in any realistic predictive use case.
2. **campaign** (number of contacts in this campaign): Could be mistaken for an outcome, but it records how many times the bank contacted the client — a property of the campaign process, not the client's decision. It is a predictor, not a target.

In [ ]:
TARGET = 'y'
print(f'Target column: {TARGET}')
print(df[TARGET].value_counts())


---
## Task 2 — Data Loading and Exploration

> *Lecture material: Lecture 1 (Problem Formulation), Lecture 2 (Data Inspection and EDA).*

The dataset contains **4,119 rows and 21 columns** (including the target). Features span four domains: demographic (age, job, marital, education), financial (default, housing, loan), campaign-related (contact, month, day_of_week, duration, campaign, pdays, previous, poutcome), and macroeconomic (emp.var.rate, cons.price.idx, cons.conf.idx, euribor3m, nr.employed).

**Numerical variables** (10): age, duration, campaign, pdays, previous, emp.var.rate, cons.price.idx, cons.conf.idx, euribor3m, nr.employed.  
**Categorical variables** (10): job, marital, education, default, housing, loan, contact, month, day_of_week, poutcome.

EDA is performed on the **full dataset before splitting** because we are only observing. There are no parameters fitted, no thresholds learned. This is safe from a leakage perspective.

### Target Distribution

The target **y** is heavily imbalanced: approximately **89% no and 11% yes**. With only ~451 positive cases out of 4,119, a model that ignores every feature and predicts no universally (the Zero Rule baseline, as covered in Lecture 11) would score ~89% accuracy while being completely useless for the actual business goal of identifying likely subscribers. This means accuracy is a misleading metric here, and stratified splitting is necessary to ensure each split reflects the true class ratio. With so few positives, an unstratified random split could leave the validation set with very few or no positive examples at all.
### Explicit and Implicit Missing Values

There are **no explicit NaN values** in this dataset. However, several categorical columns contain the string `'unknown'`, meaning the information was simply not recorded rather than being a genuine category. The affected variables are **job, marital, education, default, housing, and loan**, with **default** being the most affected (~20% unknown). These cannot simply be dropped or imputed with the mode without losing potentially meaningful signal. A client whose credit default status is unknown may be systematically different from one whose status is known.

**pdays** uses the sentinel value **999** to mean "client was not previously contacted." This is a placeholder, not a measurement. Treating it as numeric would mislead Logistic Regression into thinking pdays=999 is numerically close to pdays=998, when in reality the two represent entirely different situations (one contacted, one not).

### Variable Requiring Special Consideration

**duration** (last contact duration in seconds) correlates ~0.40 with the target, among the highest of any feature. However, it is only known after a call ends, making it unavailable at prediction time. Including it constitutes **Target Leakage**.

In [ ]:
print(f'Shape: {df.shape}')
print('\nData types:')
print(df.dtypes)
print('\nBasic statistics:')
df.describe()


In [ ]:
numerical_cols = df.select_dtypes(include=['int64','float64']).columns.tolist()
categorical_cols = [c for c in df.select_dtypes(include='object').columns if c != TARGET]
print(f'Numerical ({len(numerical_cols)}): {numerical_cols}')
print(f'Categorical ({len(categorical_cols)}): {categorical_cols}')


In [ ]:
# Target distribution
target_counts = df[TARGET].value_counts()
imbalance = target_counts.min() / target_counts.max()
fig, ax = plt.subplots(figsize=(5,4))
target_counts.plot(kind='bar', ax=ax, color=['steelblue','coral'], edgecolor='white')
ax.set_title('Target Variable Distribution')
ax.set_xticklabels(['No','Yes'], rotation=0)
ax.set_ylabel('Count')
for p in ax.patches:
    ax.annotate(str(int(p.get_height())), (p.get_x()+p.get_width()/2, p.get_height()+20), ha='center')
plt.tight_layout()
plt.show()
print(f'Imbalance ratio (minority/majority): {imbalance:.4f}')
print('Dataset is heavily imbalanced (~11% positive). Accuracy alone will be misleading.')


In [ ]:
# Explicit missing values
missing = df.isnull().sum()
print('Explicit NaN:', missing[missing>0].to_dict() if missing.sum()>0 else 'None')

# Implicit missing: 'unknown'
print('\nImplicit missing ("unknown"):')
for col in categorical_cols:
    n = (df[col]=='unknown').sum()
    if n: print(f'  {col}: {n} ({100*n/len(df):.1f}%)')

# Sentinel in pdays
print(f'\npdays=999 ("not previously contacted"): {(df["pdays"]==999).sum()} rows ({100*(df["pdays"]==999).mean():.1f}%)')


In [ ]:
# Distribution of two numerical variables
fig, axes = plt.subplots(1, 2, figsize=(12,4))
axes[0].hist(df['age'], bins=30, color='steelblue', edgecolor='white')
axes[0].set_title('Age Distribution')
axes[0].set_xlabel('Age')
axes[1].hist(df['euribor3m'], bins=30, color='coral', edgecolor='white')
axes[1].set_title('Euribor 3m Rate Distribution')
axes[1].set_xlabel('euribor3m')
plt.tight_layout()
plt.show()
print('Age: roughly bell-shaped. Euribor3m: bimodal — two distinct macroeconomic regimes in the data.')


In [ ]:
# Distribution of two categorical variables
fig, axes = plt.subplots(1, 2, figsize=(14,4))
df['job'].value_counts().plot(kind='bar', ax=axes[0], color='steelblue', edgecolor='white')
axes[0].set_title('Job Distribution')
axes[0].tick_params(axis='x', rotation=45)
df['education'].value_counts().plot(kind='bar', ax=axes[1], color='coral', edgecolor='white')
axes[1].set_title('Education Distribution')
axes[1].tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.show()


In [ ]:
# Variable requiring special consideration: duration (leakage risk)
df_temp = df.copy()
df_temp['y_bin'] = (df_temp['y']=='yes').astype(int)
corr = df_temp['duration'].corr(df_temp['y_bin'])
print(f'Correlation of duration with y: {corr:.4f}')
print('duration is highly predictive but is UNAVAILABLE at prediction time (post-call). Must be dropped.')


---
## Task 3 — Managing Missing Values (Strategy)

> *Lecture material: Lecture 2 (Data Inspection), Lecture 5 (Preprocessing and Pipeline Discipline).*

| Type | Variables | Strategy | Cleaning or Modelling? |
|------|-----------|----------|------------------------|
| Explicit NaN | None | — | — |
| Implicit `'unknown'` | `job`, `marital`, `education`, `default`, `housing`, `loan` | Retain as a **separate category** | Modelling decision |
| Sentinel `pdays=999` | `pdays` | Create binary indicator `pdays_contacted`, replace 999 with NaN, impute with **training median** | Cleaning (sentinel → NaN) + Modelling (impute) |
| Leakage variable | `duration` | **Drop entirely** | Cleaning decision |

### Cleaning vs Modelling Decisions

A **data cleaning decision** corrects a representation error that exists regardless of what the model will learn. Replacing pdays=999 with NaN is cleaning: 999 is a code, not a measurement, and leaving it as a number would actively mislead any downstream transformation.

A **modelling decision** determines how to handle missingness in a way that affects what the model learns. Keeping **unknown** as its own category is a modelling decision: the fact that a client's credit default status is unknown may itself carry predictive signal, banks may lack records on riskier or less-engaged clients. Imputing with the mode or dropping these rows would erase this signal.

### Why a Binary Indicator for `pdays`

Simply replacing 999 with NaN and imputing the median would collapse two fundamentally different situations: "contacted 5 days ago" and "never contacted" into a single continuous scale. The binary indicator **pdays_contacted** explicitly tells the model which regime each row belongs to.

### Why Fit on Training Set Only

The median used to impute **pdays** is computed exclusively from **X_train**. Computing it on the full dataset before splitting would allow the imputed value to incorporate distributional information from the validation and test sets. This creates **Preprocessing Leakage**, a subtle flaw that biases the preprocessing step toward the overall data distribution rather than the training distribution alone, producing overly optimistic metrics.

---
## Task 4 — Data Splitting

> *Lecture material: Lecture 2 (Data Splitting and Leakage), Lecture 9 (ML Pipeline).*

**Proportions:** 70% train / 15% validation / 15% test.  
With 4,119 rows this gives approximately 2,883 / 618 / 618 samples. This is sufficient for SMOTE (which needs a meaningful number of minority-class examples to learn from) and for stable metric estimates on the validation set.

**Why stratify?** With only ~11% positive class, a random split could place disproportionately few positives in the validation or test set, making precision, recall, and F1 estimates unreliable. As discussed in Lecture 8, the **stratify=y** argument in **train_test_split** guarantees each split mirrors the original 89/11 class ratio. If the test set has a different churn ratio than production, accuracy becomes meaningless.

**Why split before any transformation?** The split must occur before any transformer is fitted. If we encoded or scaled before splitting, the encoder/scaler would learn statistics (category frequencies, means, standard deviations) from the entire dataset, including the validation and test observations. This is **Preprocessing Leakage**: the model indirectly sees test-set information during training, producing optimistic metrics that do not reflect real-world performance.

**Types of leakage that would arise from late splitting:**
- **Statistical leakage**: Scalers and imputers absorb test-set distributional properties.
- **Resampling leakage**: SMOTE applied before splitting generates synthetic samples interpolated between real examples that include test-set points, effectively placing near-duplicates of test observations in the training set.
- **Selection leakage**: Feature selection based on variance or correlation computed on the full dataset may retain or discard features based on test-set patterns.



In [ ]:
X = df.drop(columns=[TARGET])
y = (df[TARGET]=='yes').astype(int)

X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.30, random_state=RANDOM_STATE, stratify=y)
X_val, X_test, y_val, y_test     = train_test_split(X_temp, y_temp, test_size=0.50, random_state=RANDOM_STATE, stratify=y_temp)

for name, X_s, y_s in [('Train', X_train, y_train), ('Validation', X_val, y_val), ('Test', X_test, y_test)]:
    print(f'{name:12s}: {X_s.shape[0]} rows | positive rate: {y_s.mean():.4f}')


---
## Task 5 — Managing Missing Values (Fit & Transform)

> *Lecture material: Lecture 5 (Preprocessing), Lecture 9 (Pipeline Discipline).*

All operations below are fit on **X_train** only, then applied identically to all three splits. This ordering is critical: fitting on any split other than the training set would allow the preprocessing to absorb information from held-out data, violating the independence assumption between training and evaluation and leading to **Preprocessing Leakage**.
Three operations are performed here:

1. **Drop duration:** Removed from all splits because it constitutes **Target Leakage**. It is not an imputation decision but a feature removal; no parameter is learned.
2. **Create pdays_contacted (Feature Synthesis):** A binary indicator (1 = client was previously contacted, 0 = not). Applied identically to all splits with no fitting required, since the rule is deterministic (**pdays ≠ 999**).
3. **Impute pdays:** The median of **pdays** among previously-contacted clients is computed from **X_train** only, then used to fill the NaN values introduced in step 2 across all splits. Using the median rather than the mean makes the imputation robust to any outliers in the contact-day distribution.

In [ ]:
# Drop 'duration' — post-call information, causes data leakage
for split in [X_train, X_val, X_test]:
    split.drop(columns=['duration'], inplace=True, errors='ignore')
print('Dropped: duration')

# Handle pdays sentinel: create indicator + replace 999 with NaN
for split in [X_train, X_val, X_test]:
    split['pdays_contacted'] = (split['pdays'] != 999).astype(int)
    split.loc[split['pdays'] == 999, 'pdays'] = np.nan
print('Created pdays_contacted indicator')

# Impute pdays with median from TRAIN ONLY
pdays_median = X_train['pdays'].median()
if np.isnan(pdays_median):
    pdays_median = 0.0
for split in [X_train, X_val, X_test]:
    split['pdays'] = split['pdays'].fillna(pdays_median)
print(f'Imputed pdays NaN with train median: {pdays_median}')

# 'unknown' in categoricals is kept as-is (handled by encoder)
print('"unknown" categories retained as explicit category for encoding.')


---
## Task 6 — Encoding Categorical Variables

> *Lecture material: Lecture 4 (Categorical Encoding), Lecture 6 (Linear Models), Lecture 9 (Feature Engineering).*

| Variable | Type | Encoding | Justification |
|----------|------|----------|---------------|
| `education` | Ordinal | `OrdinalEncoder` | Clear educational progression: illiterate → basic.4y → basic.6y → basic.9y → high.school → professional.course → university.degree |
| `month` | Ordinal | `OrdinalEncoder` | Calendar order is meaningful and preserves seasonal patterns in a single feature |
| `day_of_week` | Ordinal | `OrdinalEncoder` | Week cycle has a natural order (Mon–Fri) |
| `job`, `marital`, `contact`, `poutcome`, `default`, `housing`, `loan` | Nominal | `OneHotEncoder` (drop='first') | No intrinsic ordering; one-hot encoding gives Logistic Regression a separate, interpretable coefficient per category |

**Why drop='first' (Dummy Variable Trap)?** With k categories, OHE without dropping produces k perfectly collinear columns (they always sum to 1). This is known as the **Dummy Variable Trap**. This perfect multicollinearity makes the design matrix singular, rendering the Logistic Regression coefficient estimates undefined. Dropping one reference category removes the redundancy while preserving all information.
**Must be fit on train only to prevent Preprocessing Leakage:** The encoder must learn which categories exist from the training data only. If it learned categories from the test set, it would leak information. The **handle_unknown='ignore'** argument ensures that if a completely unseen category appears in validation or test.
### Effect on Dimensionality

Before encoding, the dataset has 20 features. After OHE, each nominal variable expands into (k−1) binary columns. For example, **job** has 12 categories and contributes 11 columns; **poutcome** has 4 categories and contributes 3. The total feature count increases substantially. This higher dimensionality is necessary for Logistic Regression to distinguish between nominal categories, but it also increases the risk of overfitting.

As discussed in Lecture 12, this expansion is exactly why Logistic Regression applies **L2 Regularization (Ridge)** by default: it penalizes large coefficients quadratically, limiting model flexibility and stabilizing estimates when dealing with the newly expanded, potentially correlated feature space.

### Effect on Coefficient Interpretability

With ordinal encoding, a single coefficient describes the effect of moving one step up the ordinal scale (e.g., one additional education level). With one-hot encoding, each coefficient describes the effect of belonging to a specific category relative to the dropped reference category. Both are directly interpretable in the context of log-odds.
### Effect on Decision Boundaries

Logistic Regression produces a linear decision boundary in the encoded feature space. Ordinal encoding allows the model to learn a monotone or non-monotone linear trend along the ordinal axis. One-hot encoding allows the model to assign independent log-odds weights to each category, which is equivalent to fitting a piecewise constant function over the nominal variable, the most expressive linear representation of categorical data.

In [ ]:
ordinal_map = {
    'education':   ['illiterate','basic.4y','basic.6y','basic.9y','high.school','professional.course','university.degree','unknown'],
    'month':       ['jan','feb','mar','apr','may','jun','jul','aug','sep','oct','nov','dec'],
    'day_of_week': ['mon','tue','wed','thu','fri']
}
ord_cols = list(ordinal_map.keys())
nominal_cols = [c for c in categorical_cols if c in X_train.columns and c not in ord_cols]

# Ordinal encoding
ord_enc = OrdinalEncoder(categories=list(ordinal_map.values()),
                         handle_unknown='use_encoded_value', unknown_value=-1)
X_train[ord_cols] = ord_enc.fit_transform(X_train[ord_cols])
X_val[ord_cols]   = ord_enc.transform(X_val[ord_cols])
X_test[ord_cols]  = ord_enc.transform(X_test[ord_cols])

# One-hot encoding
ohe = OneHotEncoder(drop='first', sparse_output=False, handle_unknown='ignore')
ohe.fit(X_train[nominal_cols])
ohe_names = ohe.get_feature_names_out(nominal_cols)

def apply_ohe(X, enc, cols, names):
    enc_df = pd.DataFrame(enc.transform(X[cols]), columns=names, index=X.index)
    return pd.concat([X.drop(columns=cols), enc_df], axis=1)

X_train = apply_ohe(X_train, ohe, nominal_cols, ohe_names)
X_val   = apply_ohe(X_val,   ohe, nominal_cols, ohe_names)
X_test  = apply_ohe(X_test,  ohe, nominal_cols, ohe_names)

print(f'Shape after encoding — Train: {X_train.shape}, Val: {X_val.shape}, Test: {X_test.shape}')


---
## Task 7 — Feature Scaling

> *Lecture material: Lecture 5 (Feature Scaling), Lecture 6 (Logistic Regression and Optimization).*

**Strategy: `StandardScaler` (Z-score standardisation)**

All numerical features are transformed to have zero mean and unit variance: z = (x − μ) / σ, where μ and σ are computed from the training set only.

### Effect on Gradient-Based Optimisation

Logistic Regression minimises a convex loss function via gradient descent (or a variant such as L-BFGS). When features are on very different scales — for example, **age** ∈ from 17 to 98 vs **emp.var.rate** from -3.4 to 1.4 vs **nr.employed** from 4900 to 5200, the loss surface becomes highly elongated. Gradient steps oscillate across the narrow dimensions while making slow progress along the wide ones. Standardisation makes the loss surface more spherical, allowing larger and more stable gradient steps and faster convergence.
### Effect on Coefficient Magnitude and Comparability

Without scaling, a coefficient reflects the change in log-odds per unit increase in the raw feature. A coefficient of 0.01 for **nr.employed** (unit = one employee) and 0.5 for **emp.var.rate** (unit = one percentage point) are not directly comparable. After standardisation, all coefficients are expressed in units of standard deviations, making them directly comparable as measures of relative feature importance.
### Effect on Regularisation

As discussed in Lecture 12, L1 and L2 penalties shrink coefficients toward zero. If features are not scaled, a feature with a small natural unit (e.g., **nr.employed**, measured in thousands) will have a large raw coefficient and be disproportionately penalised relative to a feature with a large natural unit. Standardisation ensures the regularisation penalty is applied equally to all features, making the regularisation strength parameter C interpretable and consistent across the feature set.
### Why Not MinMax Normalisation?

MinMax scales features from 0 to 1. This is sensitive to outliers: a single extreme value in **age** or a macroeconomic variable would compress the majority of values into a narrow range, reducing the effective variance of the feature. StandardScaler is more robust in this respect, which aligns with our strategies for handling outliers.


In [ ]:
scaler = StandardScaler()
X_train_s = pd.DataFrame(scaler.fit_transform(X_train), columns=X_train.columns, index=X_train.index)
X_val_s   = pd.DataFrame(scaler.transform(X_val),   columns=X_val.columns,   index=X_val.index)
X_test_s  = pd.DataFrame(scaler.transform(X_test),  columns=X_test.columns,  index=X_test.index)
print('Scaling complete.')
print('\nTrain mean (should be ~0):'); print(X_train_s.mean().round(3).head(5))
print('\nTrain std  (should be ~1):'); print(X_train_s.std().round(3).head(5))


---
## Task 8 — Feature Selection

> *Lecture material: Lecture 5 (Feature Selection), Lecture 6 (Linear Models), Lecture 9 (Pipeline Discipline).*

### Why on Train Only

Variance and correlation are statistics learned from data. Computing them on the full dataset before splitting would mean the selection criteria are informed by validation and test-set patterns, a form of **Preprocessing Leakage**. A feature that is nearly constant in the test set but informative in training could be wrongly dropped; a pair that appears correlated in the full dataset but not in training alone could be wrongly removed. Both operations must therefore use only **X_train** statistics.
### Low-Variance Filter

**Threshold = 0.01**: After **StandardScaler**, all non-constant features have variance ≈ 1 by construction. A post-standardisation variance near zero indicates a feature that was essentially constant before scaling, it carries no discriminative information. The 0.01 threshold is deliberately conservative: it removes only genuinely constant features and retains everything with meaningful variation.
### High-Correlation Filter

**Threshold = 0.90**: Pairs of features with |r| > 0.90 are nearly linearly dependent. This directly violates one of Logistic Regression's main assumptions: that features are not perfectly perfectly correlated (multicollinearity). High multicollinearity makes the model's math unstable — even tiny changes in the training data can cause massive, random swings in the feature weights. The model might still make okay predictions, but its coefficients become completely uninterpretable, which ruins our ability to explain why the model made a certain choice.
**The correlated pairs found and what they mean:**

**euribor3m — emp.var.rate** (r = 0.97): Both capture macroeconomic conditions. The Euribor 3-month rate and employment variation rate move together in economic cycles. They are essentially measuring the same underlying signal.
**nr.employed — euribor3m** (r = 0.94): Number of employees is another macroeconomic indicator that tracks the same cycle. Together, the three macroeconomic features (**emp.var.rate**, **euribor3m**, **nr.employed**) are heavily redundant — retaining one is sufficient.
**loan_unknown — housing_unknown** (r = 1.00): Perfect correlation. These two columns are identical: every client missing their loan status is also missing their housing status. This likely reflects a data collection gap where both fields were left blank together. One column contains zero additional information beyond the other.
**poutcome_success — pdays_contacted** (r = 0.93): A successful previous outcome almost always implies the client was previously contacted, and being previously contacted is almost a prerequisite for a successful outcome. These two features encode nearly the same event.
**Drop criterion**: When a correlated pair is identified, the feature with the higher mean absolute correlation across the entire feature set is dropped. This retains the feature that is most "independent" from the rest — the one that contributes more unique information relative to all other features.
### Consequence for Logistic Regression

Removing redundant features directly improves coefficient stability. With 33 features remaining (down from a peak after OHE), the design matrix is better conditioned, the L2 regularisation penalty operates more consistently, and the resulting coefficients are more interpretable.

In [ ]:
# Low variance filter
vt = VarianceThreshold(threshold=0.01)
vt.fit(X_train_s)
mask = vt.get_support()
dropped_var = X_train_s.columns[~mask].tolist()
print(f'Low-variance features dropped (threshold=0.01): {dropped_var}')
X_train_s = X_train_s.loc[:, mask]
X_val_s   = X_val_s.loc[:, mask]
X_test_s  = X_test_s.loc[:, mask]


In [ ]:
# High correlation filter
corr_matrix = X_train_s.corr().abs()
upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
high_corr = [(c, r) for c in upper.columns for r in upper.index if upper.loc[r,c] > 0.90]
print(f'Highly correlated pairs (|r|>0.90):')
for a, b in high_corr:
    print(f'  {a} — {b}: {corr_matrix.loc[a,b]:.4f}')

to_drop = set()
for a, b in high_corr:
    to_drop.add(a if corr_matrix[a].mean() >= corr_matrix[b].mean() else b)
to_drop = list(to_drop)
print(f'\nDropped: {to_drop}')
X_train_s = X_train_s.drop(columns=to_drop, errors='ignore')
X_val_s   = X_val_s.drop(columns=to_drop, errors='ignore')
X_test_s  = X_test_s.drop(columns=to_drop, errors='ignore')
print(f'Final feature count: {X_train_s.shape[1]}')


---
## Task 9 — Addressing Class Imbalance

> *Lecture material: Lecture 3 (Class Imbalance), Lecture 4 (Evaluation Metrics), Lecture 9 (Pipeline Discipline).*

### Class Distribution in the Training Set

After the 70/15/15 stratified split, the training set preserves the original ~89/11 ratio. With approximately 2,883 training samples, roughly 2,563 belong to class 0 ('no') and 320 to class 1 ('yes'). The imbalance ratio (minority/majority) is approximately 0.12, a massive imbalance that we have to fix before training.

### Why Imbalance is a Concern

Logistic Regression tries to minimize overall cross-entropy error. With 89% negative examples, the model can easily "cheat" and get a really low error score just by guessing 'no' for every single person. The math simply ignores the 11% positive class because the majority class drowns it out. Without fixing this, the model learns to say 'no' to everyone, giving us a technically high accuracy but totally failing to actually find the subscribers we care about.
### Impact on Evaluation Metrics

**Accuracy**: A model always predicting 'no' gets ~89% accuracy — which is completely useless for finding actual subscribers.
**Precision**: Can look artificially good if the model only guesses 'yes' once or twice and happens to be right, but misses the other 318 subscribers.
**Recall**: This will be near zero because the model just defaults to the majority. This is the most critical metric to watch when dealing with imbalanced data.

This is why accuracy alone is insufficient and why F1 score (which balances precision and recall) is a more appropriate metric here.

### Strategy: SMOTE

**SMOTE** (Synthetic Minority Over-sampling TEchnique) creates fake but realistic minority-class samples. It picks a minority data point, finds its nearest minority neighbors, and creates a new data point randomly placed on the line connecting them in the feature space.

**Why SMOTE over random oversampling?** Random oversampling just copy-pastes existing minority examples. This makes the model memorize those exact few points (overfitting) instead of learning the general pattern. SMOTE creates brand-new, realistic examples that help the model learn a broader, better decision boundary.

**Why SMOTE over undersampling?** We only have 320 minority examples in the training set. If we threw away majority examples to match that 320, we would only have 640 total rows left to train on, which is way too few to build a reliable model. SMOTE lets us keep all our valuable majority-class data while boosting the minority class.
### Correct Stage in the Pipeline

SMOTE is applied **after all preprocessing** (encoding, scaling, feature selection) and **only to the training set**. Applying it before preprocessing would create fake data based on raw categories, which messes up the math. Applying it to the validation or test sets would ruin the real-world 89/11 ratio we need for an honest evaluation.

**What if we used SMOTE before splitting the data?** This is a classic data leakage mistake. Since SMOTE creates fake points by blending real points, doing it on the whole dataset means a fake point in the training set might be a direct blend of a real point that ends up in the test set. The model basically gets a sneak peek at the test data during training, leading to overly optimistic and fake performance scores.

In [ ]:
print('Class distribution in training set (before SMOTE):')
print(y_train.value_counts())

smote = SMOTE(random_state=RANDOM_STATE)
X_train_res, y_train_res = smote.fit_resample(X_train_s, y_train)

print('\nAfter SMOTE:')
print(pd.Series(y_train_res).value_counts())
print(f'\nValidation set preserved: {y_val.value_counts().to_dict()}')
print(f'Test set preserved:       {y_test.value_counts().to_dict()}')


---
## Task 10 — Training a Logistic Regression Model

> *Lecture material: Lecture 6 (Logistic Regression), Lectures 9–11 (Model Evaluation).*

We train a Logistic Regression with **solver='lbfgs'** and **max_iter=1000** on the SMOTE-resampled training set, then evaluate it on the **untouched validation set**.
### Why Logistic Regression

The assignment specifically requires Logistic Regression. Beyond just following the rules, it is a great baseline model to test our pipeline. Even though it's limited to drawing straight lines (linear decision boundaries), it is fast, highly interpretable (the coefficients tell us exactly how much a feature changes the odds), and if it fails to beat a simple baseline, we know the problem is with our data prep, not some complex "black box" algorithm acting up.

### Solver and Convergence

**solver='lbfgs'** is the default optimization algorithm in scikit-learn. It works great for small-to-medium datasets that use L2 regularization. However, the default **max_iter=100** (the maximum number of steps it can take to find the bottom of the "loss valley" we talked about earlier) isn't enough here. Because SMOTE basically doubled our training data by adding synthetic minority samples, the algorithm needs more steps to settle on the best answer. Bumping it to **max_iter=1000** just gives it enough time to finish the math properly without changing the model itself.
### Default L2 Regularisation

Logistic Regression in scikit-learn applies L2 regularization by default (**C=1.0**). As we discussed earlier, this shrinks all the feature weights toward zero so no single feature takes over, which prevents overfitting. This is especially crucial now that SMOTE has injected fake data points. We are sticking with the default **C** value because tuning it would require cross-validation, which is outside the scope of this preprocessing-focused assignment.
### Zero-Rule Baseline

We use a **DummyClassifier** that simply guesses the majority class every single time. Since our validation set keeps the real-world 89/11 split, this dummy model will just guess 'no' for everyone. It will get ~89% accuracy, but find exactly zero actual subscribers (0% recall). This proves exactly why we have to look at Recall and F1-score. If our real model gets an accuracy much higher than 89% on this imbalanced set, we should actually be suspicious. It might just mean it's predicting 'yes' even less often. Any useful model must beat this baseline on recall and F1, not accuracy.
### Why Evaluate on Validation, Not Test

The test set is reserved for a final, unbiased estimate of generalisation performance after all modelling decisions are finalised. Using it during development, even just to look at the metrics, would cause us to unconsciously tune decisions toward the test set. The validation set exists precisely to make iterative evaluation possible without contaminating the test set.

In [ ]:
# Zero-Rule Baseline
dummy = DummyClassifier(strategy='most_frequent', random_state=RANDOM_STATE)
dummy.fit(X_train_res, y_train_res)
y_dummy = dummy.predict(X_val_s)
dummy_acc  = accuracy_score(y_val, y_dummy)
dummy_prec = precision_score(y_val, y_dummy, zero_division=0)
dummy_rec  = recall_score(y_val, y_dummy, zero_division=0)
dummy_f1   = f1_score(y_val, y_dummy, zero_division=0)

# Logistic Regression
lr = LogisticRegression(max_iter=1000, random_state=RANDOM_STATE, solver='lbfgs')
lr.fit(X_train_res, y_train_res)
y_pred = lr.predict(X_val_s)
lr_acc  = accuracy_score(y_val, y_pred)
lr_prec = precision_score(y_val, y_pred, zero_division=0)
lr_rec  = recall_score(y_val, y_pred, zero_division=0)
lr_f1   = f1_score(y_val, y_pred, zero_division=0)

results = pd.DataFrame({
    'Model':     ['Zero-Rule Baseline', 'Logistic Regression'],
    'Accuracy':  [dummy_acc,  lr_acc],
    'Precision': [dummy_prec, lr_prec],
    'Recall':    [dummy_rec,  lr_rec],
    'F1':        [dummy_f1,   lr_f1]
}).round(4)
print('=== Validation Metrics ===')
print(results.to_string(index=False))


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Confusion Matrix
cm = confusion_matrix(y_val, y_pred)
ConfusionMatrixDisplay(cm, display_labels=['No','Yes']).plot(ax=axes[0], colorbar=False, cmap='Blues')
axes[0].set_title('Confusion Matrix — Validation Set', fontsize=13)

# Bar chart comparison
x = np.arange(4)
w = 0.35
dummy_scores = [dummy_acc, dummy_prec, dummy_rec, dummy_f1]
lr_scores    = [lr_acc,    lr_prec,    lr_rec,    lr_f1]
axes[1].bar(x-w/2, dummy_scores, w, label='Zero-Rule',          color='blue', edgecolor='white')
axes[1].bar(x+w/2, lr_scores,    w, label='Logistic Regression', color='steelblue', edgecolor='white')
axes[1].set_xticks(x)
axes[1].set_xticklabels(['Accuracy','Precision','Recall','F1'])
axes[1].set_ylim(0, 1.15)
axes[1].set_ylabel('Score')
axes[1].set_title('Validation Metrics Comparison', fontsize=13)
axes[1].legend()
for bar in axes[1].patches:
    h = bar.get_height()
    if h > 0.01:
        axes[1].text(bar.get_x()+bar.get_width()/2, h+0.01, f'{h:.2f}', ha='center', va='bottom', fontsize=9)
plt.tight_layout()
plt.show()


### Interpretation

**Accuracy** drops slightly below the Zero-Rule baseline; however, this is an expected and theoretically sound result. While the Zero-Rule achieves ~89% accuracy by predicting 'no' for every instance, it fails to provide any discriminative utility. Our model, trained on **SMOTE-balanced data**, identifies positive cases ('yes'). As seen in the Confusion Matrix, the model successfully identified 45 subscribers (True Positives) that the baseline would have missed. While this naturally increases the risk of False Positives (104 cases) and lowers raw accuracy on an imbalanced validation set, it represents the correct optimization for a classification problem where the minority class is the primary focus.

**Recall** is substantially higher than that of the Zero-Rule. This indicates the model is successfully identifying clients likely to subscribe rather than defaulting to the majority class. From a business perspective, this shift is critical: it allows a marketing campaign to be targeted at a high-probability shortlist of potential subscribers, significantly improving resource allocation compared to indiscriminate contact.

**F1-Score** confirms that the model is capturing genuine predictive signals from the features. By balancing the precision-recall tradeoff, the F1-score provides a more honest and reliable summary of model performance than accuracy when dealing with extreme class imbalance.

A lower accuracy compared to the Zero-Rule baseline is not a failure of the model. Instead, it is a direct consequence of prioritizing **Recall** on the minority class, the fundamental objective of using SMOTE. This tradeoff demonstrates that the model is effectively learning to distinguish the characteristics of subscribers, even within a highly skewed distribution.

---
## Conclusion

The pipeline progresses from raw data to a trained, evaluated classifier through ten stages, each structured to prevent **Preprocessing Leakage** and respect the mathematical assumptions of Logistic Regression.

The primary decisions and justifications are as follows: **duration** was removed due to **Target Leakage**, as it is unavailable at the time of prediction. The **pdays** sentinel value was decomposed into a binary indicator and an imputed numeric value to distinguish between qualitatively different states (contacted vs. not contacted). Categorical variables were encoded using ordinal or one-hot schemes based on their intrinsic properties, with **drop='first'** applied to eliminate  multicollinearity in the design matrix. Features were standardized using **StandardScaler** to balance the gradient landscape and ensure the **L2 regularization** penalty is applied consistently across all features. Redundant features were removed via a high-correlation filter to stabilize coefficient estimates. Finally, **SMOTE** was applied strictly to the training set after all preprocessing to augment the minority class without corrupting the evaluation sets.

The validation results confirm the pipeline's coherence: **Recall** improves significantly over the **Zero-Rule baseline**, indicating that the model is capturing predictive patterns rather than merely defaulting to the majority class. Although accuracy is slightly lower than the baseline, this is the expected trade-off when predicting a minority class within an imbalanced distribution.

Final optimization steps would include **Threshold Tuning** (as the default 0.5 threshold is rarely optimal for imbalanced data), a hyperparameter search for the regularization strength (**C**) via **Cross-Validation**, and the final unbiased evaluation on the **Test Set** once all modeling decisions are finalized.

---
## Pipeline Summary

```
Raw Data
  ├── [1] Identify Target (y)
  ├── [2] EDA: distributions, missing values, leakage candidates
  ├── [3] Missing value strategy defined (no parameters learned yet)
  ├── [4] SPLIT → Train 70% / Val 15% / Test 15% (stratified on y)
  │         ↓              ↓              ↓
  │       Train           Val           Test
  ├── [5] Drop 'duration'; pdays_contacted indicator; impute pdays (train median)
  ├── [6] OrdinalEncoder.fit(train) → transform all three splits
  │        OneHotEncoder.fit(train) → transform all three splits
  ├── [7] StandardScaler.fit(train) → transform all three splits
  ├── [8] VarianceThreshold.fit(train) → select features
  │        Correlation filter (computed on train) → drop redundant features
  ├── [9] SMOTE → resample TRAIN ONLY (Val/Test unchanged)
  └── [10] LogisticRegression.fit(train_resampled) → evaluate on Val
```

All learned parameters are computed from the **training set only**. The test set is never touched until final evaluation.